##导入相关包

In [ ]:
import pandas as pd
import numpy as np
import jieba  #jieba分词
import os   
from sklearn.feature_extraction.text import TfidfVectorizer #TF-IDF
from sklearn.model_selection import GridSearchCV            #网格搜索
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix,classification_report #预测效果的评估
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['SimHei']  #这两行用于plt图像显示中文，否则plt无法显示中文
plt.rcParams['axes.unicode_minus']=False

##簡轉繁

In [ ]:
!pip3 install opencc-python-reimplemented

#数据读取(簡->繁)

In [ ]:
# 使用pandas读取数据集
import pandas as pd
# train = pd.read_table('train.txt', sep='\t',header=None)  # 训练集 占RAM太多了，當TFIDF矩陣的時候：(news_tfidf.toarray())
train = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/train.txt', sep='\t', header=None, nrows=20000)  # 训练集'dev.txt'，仍占用太多，限制只讀20000筆
test = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/test.txt', sep='\t',header=None, nrows=20000) # 测试集'test.txt'，仍占用太多，限制只讀20000筆

In [ ]:
# 添加列名便于对数据进行更好处理
train.columns = ["text_a",'label']
# dev.columns = ["text_a",'label']
test.columns = ["text_a"]

In [ ]:
# 定义要进行分类的14个类别
label_list=list(train.label.unique())
print(label_list)
print(len(label_list))  # 一定要14个类别

In [ ]:
# 查看训练集
train

In [ ]:
# 查看训练集
train["text_a"][0]="""活動資訊
課程將帶領學員製作五顆脆皮菠蘿泡芙，每個泡芙約8.5cm*8.5cm高。

情人節有double date計畫的朋友，很適合帶去分享你的幸福呦♥

 

課程亮點✪1  【不怕失敗・跟著老師一起動手】

專業甜點師帶你一起做，沒有烘焙經驗也不怕，跟著老師一步一步，做出你的完美泡芙！

課程亮點✪2 【實用的進階口味變化】

學習兩種卡士達口味（原味、巧克力），以及兩種鮮奶油（原味、草莓）的製作方式，幫助學員了解自行製作時口味可以如何進行變換，學會了你也可以是泡芙大師！

 

 

║課程內容║ 脆皮菠蘿泡芙配方講解與示範與實作、泡芙麵團判斷技巧、酥皮泡芙製作 、卡式達口味變化、鮮奶油擠花技巧

 

║課程日期║

第一場次｜㋁ ㏪ ㊅ 1300 - 1700

第二場次｜㋁ ㏫ ㊐ 1300 - 1700

 

║課程費用║每班最多8人 $2,000（精緻小班，含師資、配方食譜及材料費用）

 

║課程地點║ 臺北市中山區民族東路72號 Kitchen Starter 實驗x廚房
 

║小提醒║當日製作的泡芙可以帶回家，但為響應環保，教室沒有免費提供外帶的盒子，建議學員帶自己的容器前來，或是現場也可以加價購買$50一組的蛋糕盒呦！"""

In [ ]:
# 查看训练集
train["text_a"][1]="""學員上課心得分享

之前也透過公司的線上及線下實體課程上過一些些Python，但還是不知道Python要怎麼用，在一次FB的廣告看到了happycoder的共學營，覺得滿符合自己的需求，猶豫再三後報名了。先來說說前兩堂，也來記錄一下自己有認真的上課。每堂課的難度很適合我這種沒耐心的學生，跟我之前上過的課比起來，量不多，搭配習題，會讓人有種，咦，我會了。

記錄自己開始自學Python 的日子

這是我第2個學Python的線上平台課程，因為當時不知道有什麼樣的專案可以練習，也茫然無措。剛好發現有這個共學營，就這樣開啟了約2個月時間的Python寫程式之旅。它是每週3次發布課程內容，不要小看這個頻率，一次發佈的內容，至少需要2小時來完成作業（因為有申論題跟實作題）。也因為這樣的模式，讓我可以打好基本功，從中累積自己寫程式的能力。

8種Python自學資源推薦，「非」工程背景也能輕鬆學！
 

報名程式設計共學營前建議請先詳閱相關內容說明，確認符合學習需求。如對程式設計共學營報名相關問題有疑問可以參閱 FAQ 程式設計共學營常見問題，或是麻煩來訊我們的 FB 臉書絲專頁 ，我們將儘速回覆您，謝謝～

 

若無程式設計背景和基礎，想要學習 Python 程式語言基礎進行自動化應用和解決工作生活重複性問題，建議可以從 Python 程式設計入門
若已有部分基礎，希望學習 SQL 資料分析和網路爬蟲，可以學習 Python/SQL 商業資料分析 & 視覺化與網路爬蟲入門（課程中仍會複習 Python 重點語法但較為緊湊）
若已有部分基礎，希望結合財務分析和 LINE 聊天機器人應用，可以考慮 Python 證券投資分析 & 股票聊天機器人入門共學營（課程中仍會複習 Python 重點語法但較為緊湊）
若已有部分基礎，希望廣泛學習電腦科學基礎，學習常見不同程式語言概念，可以學習電腦科學概論 & 程式設計思維入門
為什麼要學 Python？

學習曲線和緩：語法簡潔好閱讀，適合初學者入門學習和啟發學習興趣跟獲得成就感
應用範圍廣：從電腦遊戲、資料科學與人工智慧應用、Web 網頁應用程式、網路爬蟲等都可以看見 Python 身影
CP 值高：即便不成為工程師/程式設計師，也可以透過 Python 自己設計小程式自動化生活重複、瑣碎的事情（ex. 自動抓取股票財報資料、自動訂票、自動操作 excel、分析客戶關係資料等）
 

活動適合對象

買了一堆程式書和線上課程但覺得內容過多無法消化的學習者
非資訊相關科系但對於程式設計有興趣的一般民眾想先試試看是否合適者
想要自學程式設計但不得其門而入的自學者
具備基礎電腦操作能力但想要建立基本程式設計觀念和撰寫簡單生活應用的學習者
從事 PM 產品經理或是 UI/UX 使用者經驗研究但對於程式設計有興趣的專業工作者
曾經或正在自學程式設計卻因為沒有導師和戰友可以交流討論，最後放棄的學習者
本共學營不適合：已經掌握 Python 程式開發基礎和專案開發能力的同學
 

活動預期收穫

建立基本 Python 程式設計觀念和除錯技巧，打好未來學習進階課程基礎
學習電腦科學和程式設計背後運作的邏輯，當家人朋友甚至小孩問你時不會驚慌失措
累積 4 大專題作品實作經驗，建立基本程式設計應用能力
學習如何撰寫個人技術文章，強化記憶並養成技術文章部落格寫作習慣
認識一群志同道合的戰友、學伴和同學，組成讀書會或共學團隊
透過導師/助教解惑去除自學程式單打獨鬥的學習盲點
建立未來自學程式設計和跨領域自學能力並對未來自學程式設計方向更加了解
 

活動事前準備

本共學營不需程式設計基礎，但需要具備基本電腦操作和上網能力以及中文、英文查詢資料閱讀能力
請安排共學營期間一週至少約有 6-12 小時（根據學員程度不同花的時間也有所不同），可以撰寫作業任務和學習相關內容（共 8 + 4 週 24 堂，含 4 週期末專題實作）
本共學營採線上進行，準備一個可以上網的電腦，不限作業系統（課程主要將以 Windows/MacOS 講解，但會補充 Linux 操作方式）就可以隨時隨地學習
本共學營會使用 Python3 程式語言講解
請先註冊全球最大程式設計程式碼分享平台 Github 個人帳號
 

課程活動進行方式

本期程式設計共學營課程為實作導向（非一般純影片教學），共 8 + 4 週 24 堂課程，課程活動開始後每週釋出 3 次課程內容（本期程式設計共學營為定時定量學習，非一次釋出所有課程學習內容，若有報名問題歡迎來訊詢問），最後 4 週為期末專題實作（採線上學習方式進行，課程訂閱註冊成功後不限時間地點皆可於課程網站重複學習）。每週根據頻率釋出課程內容（課程內容包括圖文內容、詳細步驟說明、範例程式碼、即時互動程式練習和補充參考文件等，非一般純影片教學）和作業任務（問答申論/程式實作題）
課程更新頻率：課程活動開始後每週二、四、六釋出當天課程學習內容和作業任務
每堂課程結束皆需完成提交作業任務至平台（提供建議繳交時間當作學習步調參考，期末專題結束前為最後補繳交時間，讓學員可以自由安排時間線上學習）。作業任務形式包含問答申論/程式實作題、專案實作、期末專題等。期末專題作業任務將採自行提案組隊方式進行
任務提交符合資格者可申請電子結業證書
課程活動平台提供課程問題討論區功能（共學營導師團將於問題討論區協助學員問題解惑，共學營活動期間導師將於課程問題討論區協助問題解析，營期結束後以學員間社群討論為主）透過社群共學機制可以認識志同道合的戰友和夥伴，活動並鼓勵學員自行組成團隊和線上下讀書會，官方討論以社群共學討論區為主
本課程活動為線上學習翻轉教育學習模式，鼓勵主動自我學習和社群共學，若為有參考答案類型的作業任務會提供參考解答和彙整學員常見錯誤及程式碼可以改善部分建議提醒說明，但不會提供個別學員細部批改和訂正通知，專案實作則會提供學員可以改善的方向和建議。若學員參考完發現仍有疑問，可以透過問題討論區留言討論，也可以透過觀摩優秀同儕作品教學相長
根據學員反饋持續更新升級課程內容和優化課程平台系統，以提供學員更優質的學習環境和使用者體驗
課程活動期間將不定期推播課程活動相關資訊激勵學員學習
為提升共學營學習品質，設有報名人數限制，額滿為止，報名從速
 

獎勵措施

結業學員獎（於課程活動結束前完成作業任務比例至少 2/3）：可申請電子結業證書乙張和虛擬紀念徽章
活動報名須知

本期早鳥優惠截止時間：2023/02/21
本期課程活動時間：2023/02/21 - 2023/05/13（本期程式共學營使用 Accupass 活動通平台進行報名繳費，報名頁面顯示活動時間為最後報名時間非課程活動時間）
本期最晚報名時間：2023/02/28
單人早鳥優惠票（回饋優惠價） NT 1,680
雙人早鳥優惠票（回饋優惠價） NT 2,960
單人票（回饋優惠價） NT 1,980（原價：2,980）
雙人票（回饋優惠價） NT 3,760（原價：5,960）
本期使用 Accupass 活動通報名系統，報名後約 1-3 個工作天會寄出報名完成通知，正式開營前將寄出兌換課程代碼
如需申請退費，本課程活動委託 Accupass 活動通代為處理退款事宜（退款說明請參閱活動通官方說明），依退款規則辦理。如需申請退款請於活動票券有效開始日前 8 日辦理，並將酌收票價 10% 退票手續費，逾期恕不受理
本共學營為線上虛擬商品服務（若已成功兌換課程或超過退費時間恕不受理退費，若需試閱請來訊粉專聯絡我們）
我們將於活動正式開始前 3 天寄出兌換課程兌換碼（活動開始後才報名者於報名後約 1-2 個工作天寄出），報名後請留意您的信箱通知
請問企業訓練是否可以發票報帳？可以，我們提供三聯式雲端電子發票。請務必於活動報名表單發票備註欄提供公司統一編號和公司抬頭，若無填寫則視為一般二聯式雲端電子發票開立，無法重複開立發票，請特別注意。發票將於課程活動最後報名結束後陸續開立。請注意雙人團報無法分開開立統一開票，若有分開開立發票需求請個別報名。
 

常見問題 FAQ

課程活動期間若有幾天需要出國旅行或出差，請問適合參與這次課程活動嗎？
可以的！程式設計共學營課程共 8 + 4 週 24 堂課（最後 4 週為期末專題實作，可單獨組隊也可以於課程討論區提案和同學一起組隊）採線上方式進行（不限時間地點皆可學習，若已成功訂閱註冊課程之後即可於課程網站重複使用內容）。每週根據頻率釋出課程內容（課程內容包括圖文內容、詳細步驟說明、範例程式碼、即時互動程式練習和補充參考文件等，非一般純影片教學）和作業任務（問答申論/程式實作題）。每堂課程結束後皆有建議學習和作業任務提交時間，期末專題為最後補繳交時間，任務形式包含課後作業（問答申論/程式實作題）、專案實作、期末專題等。導師解惑指導和知識服務時間主要為營期活動期間，營期結束後將以同儕社群共學討論為主。
 

請問課程活動需同時在線上嗎？
不需要。程式設計共學營課程是線上實作導向課程，沒有幾點幾分上課問題，當週課程學習內容釋出後（課程活動開始後預計每週二、四和六中午 12：00 釋出當天課程學習內容和作業任務）可以自行安排時間彈性線上學習。
 

請問期末專題進行方式是？
請自行規劃、需求分析並實作一個期末專題（為一次作業任務），專案內容可以是一個 程式專題應用或是延伸課程範例應用、甚至是一個整合的小專案（請評估工作量和時間分配，題目建議大小適中為宜）。期末專題採 1-4 人一組提交作業任務，可於課程討論區提案組隊，若無團報者其餘視為個人組成團隊。
 

請問課程的總時數和每次更新單元的長度？
本期程式共學營課程為實作導向學習模式，非影片錄製教學，故無總時數。課程內容包括課程圖文內容、詳細步驟說明、範例程式碼、即時互動程式練習和補充參考文件等，非一般純影片教學）和作業任務（問答申論/程式實作題）。但一般會建議學員預留每堂至少 2-3 小時學習和完成作業任務（根據學員背景程度不同增減），我們用看的學不會，相信唯有動手實作，程式學習才能學的好。

 

 

請問報名後如何兌換註冊課程？
成功報名繳費後將收到報名繳費成功通知信，我們將於課程活動正式開始前 1-3 個工作天寄出標題為課程兌換碼和課程網址的通知信，請參閱課程通知信中教學完成課程兌換。若無收到請先檢查 yahoo/hotmail 垃圾信件夾或 gmail 促銷內容信件夾，若無則請來訊我們來協助您。
 

課程上課學習時間？
本課程活動採線上方式進行，若已成功兌換課程即可於課程網站使用課程內容，課程活動時間結束後也可以根據自己時間安排繼續學習使用課程內容。課程每週根據頻率更新課程內容，可以自行安排學習時間，不限時間地點皆可學習，沒有幾點幾分上課時間的規定。但有提供學習建議時間可以參考作為作業任務和學習時間安排方向，符合作業任務提交規定則者可以申請電子結業證書。導師解惑指導和知識問答服務時間主要為營期活動期間，課程活動結束後將以同儕社群共學討論為主。
 

課程活動結束後是否可以繼續使用課程內容？
可以，程式設計共學營課程共 8 + 4 週 24 堂課（含期末專題實作）採線上方式進行（不限時間地點皆可學習，課程活動結束後仍可於課程網站重複使用存取課程內容，老師將於課程活動時間提供問題討論解惑服務，課程活動結束後將以同儕社群討論為主）。
 

作業任務批改訂正方式？
本課程活動為線上學習翻轉教育學習模式，鼓勵主動自我學習和社群共學，若為有參考答案類型的作業任務會提供參考解答和彙整學員常見錯誤及程式碼可以改善部分建議提醒說明，但不會提供個別學員細部批改和訂正通知，專案實作則會提供學員可以改善的方向和建議。若學員參考完發現仍有疑問，可以透過問題討論區留言討論，也可以透過觀摩優秀同儕作品教學相長。
 

課程活動開始後還可以報名嗎？
可以的！我們設有最後報名時間，只要在期限內報名都可以。課程內容採線上進行，不限時間地點皆可學習，若已成功訂閱課程即可於課程網站不限次數重複使用內容。為了讓學員可以彈性安排學習時間，期末專題結束前為最後作業任務補繳交時間。
 

請問海外學員適合參與這次課程活動嗎？
可以的。本課程活動主要以線上學習和線上繳交作業任務、專題實作為主，但鼓勵同學組成線上線下讀書會和期末專題分組實作討論。所以只要準備好電腦和網路並具有基礎電腦、上網操作能力的學員即可報名學習。
 

請問雙人團報該如何報名呢？
請推派一人為代表在報名區選擇雙人團報票種並註記團報人的姓名和聯絡信箱 Email，完成報名繳費。活動正式開始前將分別寄出課程兌換碼（若為活動開始後報名則於報名後 1-2 工作天寄出）。
請注意雙人團報無法分開開立統一開票，若有分開開立發票需求請個別報名。
 

請問企業訓練是否可以發票報帳？
可以，我們提供三聯式雲端電子發票。請務必於活動報名表單發票備註欄提供公司統一編號和公司抬頭，若無填寫則視為一般二聯式雲端電子發票開立，無法重複開立發票，請特別注意。發票將於課程活動最後報名結束後陸續開立。
請注意雙人團報無法分開開立統一開票，若有分開開立發票需求請個別報名。
本程式設計共學營課程無勞動部等政府補助計畫。
 

如何在 ezPay 電子發票加值服務平台列印三聯式電子發票報帳？
* 若該張發票為三聯式發票，則買受人可於 ezPay 電子發票官網「幫助中心單元/電子發票查詢專區」，輸入電子發票通知信中的發票號碼、發票日期、隨機碼及圖型驗證碼，點選「開始查詢」，接著再選取「列印電子發票」按鈕，即可列印電子發票證明聯。
注意事項：二聯式發票因有重複兌獎的後續問題，ezPay 電子發票加值服務平台僅供買受人查詢發票資訊，無法提供買受人列印服務。"""

In [ ]:
# 復原train_tmp用，便實驗
train_tmp=train.copy
# 查看训练集
train

##遍歷 train 變數的每一行，然後將每一行的簡體字轉換為繁體字

In [ ]:
from opencc import OpenCC

cc = OpenCC('s2tw')
for index, row in train.iterrows():
  train.loc[index, 'label'] = cc.convert(str(row['label']))
  train.loc[index, 'text_a'] = cc.convert(row['text_a'])
for index, row in test.iterrows():
  #test.loc[index, 'label'] = cc.convert(str(row['label']))
  test.loc[index, 'text_a'] = cc.convert(row['text_a'])

In [ ]:
tmp1=train.copy()
# 查看训练集
train

##统计

In [ ]:
# 总类别标签分布统计
train['label'].value_counts()

In [ ]:
# 文本长度统计分析,通过分析可以看出文本较短，最长为48
train['text_a'].map(len).describe()

In [ ]:
# 对测试集的长度统计分析，可以看出在长度上分布与训练数据相近
test['text_a'].map(len).describe()

In [ ]:
#待研究!
# 全局设置解决matplotlib中文显示错误问题，参考：https://aistudio.baidu.com/aistudio/projectdetail/1658980
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

import matplotlib.font_manager as font_manager

# 设置显示中文
matplotlib.rcParams['font.sans-serif'] = ['FZSongYi-Z13S'] # 指定默认字体
matplotlib.rcParams['axes.unicode_minus'] = False # 解决保存图像是负号'-'显示为方块的问题
# 设置字体大小
matplotlib.rcParams['font.size'] = 16

# 可视化类别标签分布情况
train['label'].value_counts(normalize=True).plot(kind='bar');

#X
#尚無停用詞(檔)
with open('stopwords.txt','r',encoding='utf-8') as f:
    stopwords = f.readlines()
stopwords = [i.strip() for i in stopwords]

#X
#结巴分词并去除停用词
def jieba_stop(each):
    result = []
    jieba_line = jieba.lcut(each)  #jieb.cut()返回迭代器，节省空间；jieba.lcut() 直接返回分词后的列表。三种分词模式：默认精确模式，用于文本分析  
    for l in jieba_line:
        if (l not in stopwords) and (l != ''): # 去除停用词以及空格
            result.append(l.strip())
    return result
xx = train.apply(jieba_stop)  #注意 apply 的用法
# date_news.news = date_news.news.apply(lambda x: ','.join([i for i in x if i !=''])) #反复去除空字符

In [ ]:
train=tmp1
#使用jieba
# import jieba
# for index, row in train.iterrows():
#     # 對 row['text_a'] 進行分詞
#     words = jieba.cut(row['text_a'])
#     # 将分詞结果转换为字符串，并存回 train 變量中
#     train.loc[index, 'text_a'] = ','.join(words)
#============================================================
#使用ckip_client
# import ckip_client
# for index, row in train.iterrows():
#   # 對 row['text_a'] 進行分詞
#   words = ckip_client.segment(row['text_a'])
#   # 将分詞结果转换为字符串，并存回 train 變量中
#   train.loc[index, 'text_a'] = ','.join(words)
#=======================================================
# 直接使用主題模型提取特徵字
# 導入所需模塊
from gensim import corpora, models

# 分類的label
label_list = ['彩票', '財經', '遊戲', '家居', '房產', '星座', '體育', '股票', '時政', '科技', '娛樂', '社會', '教育', '時尚']

# 分類的文本
text_list = train['text_a'].tolist()

# 定義停用詞
original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
  stop_words = f.read().split("\n")
stop_words += original_stop_words
# print(stop_words)
# # 載入停用詞至jieba
# for word in stop_words:
#     jieba.add_word(word)
#斷詞
# texts = [[word for word in jieba.cut(document)] for document in text_list]
# 精确模式(default)
texts = [[word for word in jieba.cut(document) if word not in stop_words] for document in text_list]
# 全模式
texts2 = [[word for word in jieba.cut(document, cut_all=True) if word not in stop_words] for document in text_list]
# 搜索引擎模式
texts3 = [[word for word in jieba.cut_for_search(document) if word not in stop_words] for document in text_list]



# 建立詞袋字典
dictionary = corpora.Dictionary(texts)

# 建立詞袋資料集
corpus = [dictionary.doc2bow(text) for text in texts]

# 建立LDA模型
lda = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=len(label_list))

# 使用下面的方式，指定每個主題的名稱
for i, topic in enumerate(lda.show_topics(num_topics=len(label_list), num_words=10)):
  print('主題{}:{}'.format(label_list[i], topic))

# 註: 上述程式碼中的num_topics參數可以根據你的需求調整，num_words參數可以決定每個主題中顯示的詞語數量。

In [ ]:
texts

In [ ]:
train["text_a"][1]  #原文

In [ ]:
print(texts[1])   # 精确模式(default)
print(texts2[1])  # 全模式
print(texts3[1])  # 搜索引擎模式

##分詞(ckiptagger) 目前僅 tain有分詞.test ,dev尚無
未來可用停用詞

CKIP ckiptagger 相關
pip install -U ckiptagger[tf,gdown]
pip install -U ckiptagger
pip install -U ckiptagger[tfgpu,gdown]
pip install -U ckiptagger[tfgpu,gdown]

In [ ]:
pip install ckiptagger

In [ ]:
tmp1=train.copy()
tmp1=train
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER

# Download data for ckiptagger
data_utils.download_data_gdown("/content/drive/MyDrive/CKIP/")#./ /content/drive/
# path='/content/drive/MyDrive/data/data'
# Load data for ckiptagger
ws = WS("/content/drive/MyDrive/CKIP/data")#"./data" #/content/drive/MyDrive/data
pos = POS("/content/drive/MyDrive/CKIP/data")
ner = NER("/content/drive/MyDrive/CKIP/data")

for index, row in train.iterrows():
    # 對 row['text_a'] 進行分詞
    words = ws([row['text_a']])
    # 将分詞结果转换为字符串，并存回 train 變量中
    train.loc[index, 'text_a'] = ','.join(words[0])


In [ ]:
commaSeparatedTrain=train#以空格分詞

In [ ]:
#commaSeparatedTrain.text_a = commaSeparatedTrain.text_a.apply(lambda x: ','.join([i for i in x if i !=''])) #反复去除空字符

In [ ]:
train

In [ ]:
commaSeparatedTrain